# Phase A — TMDB API ile Dengeli Veri Toplama

Resmî TMDB API ile **dengeli + kombinasyon-farkında** film/poster çeker (`src/data/tmdb_api.py`); HTML scraping'in ban sorununu çözer.

**İki faz:** (1) tür bazlı aday havuzu, (2) eksiklik-güdümlü greedy seçim (baskın türler "yolcu" olarak en sona, overshoot azaltılır).

**Çıktı:** `DATA_ROOT/labels_v2.csv` + `DATA_ROOT/posters/{tmdb_id}.jpg`

## 0) Kurulum

In [ ]:
import os, sys
from pathlib import Path

# --- Yerel ---
CODE_ROOT = Path('../')
DATA_ROOT = Path('../')

# --- Colab (yukaridaki ikisini yorumlayip bunlari ac) ---
# !pip -q install requests pandas matplotlib
# from google.colab import drive
# drive.mount('/content/drive')
# CODE_ROOT = Path('/content/drive/MyDrive/film-genre-project')
# DATA_ROOT = Path('/content/drive/MyDrive/film-genre-project-data')
# from google.colab import userdata
# os.environ['TMDB_API_KEY'] = userdata.get('TMDB_API_KEY')   # Secrets'a ekledikten sonra

# Yerelde proje kokundeki .env otomatik okunur (TMDB_API_KEY / TMDB_BEARER).
sys.path.insert(0, str(CODE_ROOT.resolve()))
from src.data.tmdb_api import collect_balanced, download_posters, save_labels, SCARCITY_ORDER

has_key = bool(os.environ.get('TMDB_API_KEY') or os.environ.get('TMDB_BEARER') or (CODE_ROOT / '.env').exists())
assert has_key, 'TMDB anahtari yok: Colab Secrets ekle veya .env doldur'
print('Kurulum OK | DATA_ROOT =', DATA_ROOT.resolve())

## 1) (Opsiyonel) Smoke test — API + denge mantigini 1 dk'da dogrula

In [ ]:
# Kucuk, postersiz cekim — sadece dogrulama
df_smoke = collect_balanced(per_genre=50, min_votes=30)
print()
print('Smoke test satir:', len(df_smoke))
df_smoke.head()

## 2) Tam çekim

Her tür ~2500 hedef (fine-tune için yeterli; daha fazla kombinasyon kapsaması istersen `PER_GENRE`'yi artır — eğitim maliyeti de artar). Rare türler (History vb.) havuzda yetersiz kalırsa `MIN_VOTES`'u düşür veya `POOL_FACTOR`'u artır.

In [ ]:
PER_GENRE   = 2500   # tur basina hedef
MIN_VOTES   = 30     # az oylu/kalitesiz filmleri eler; rare turler eksikse 10-20'ye dusur
POOL_FACTOR = 1.5    # rare turler hedefe ulasmiyorsa 2.0-3.0 dene

df = collect_balanced(per_genre=PER_GENRE, min_votes=MIN_VOTES, pool_factor=POOL_FACTOR)
save_labels(df, DATA_ROOT / 'labels_v2.csv')
print('Toplam film:', len(df))

## 3) Poster indirme

In [ ]:
ok = download_posters(df, DATA_ROOT / 'posters', size='w500', workers=16)
print(ok, '/', len(df), 'poster indirildi ->', DATA_ROOT / 'posters')

## 4) Çekim öncesi/sonrası dağılım (rapor / vize figürü)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

def genre_counts(csv_path):
    d = pd.read_csv(csv_path, dtype={'tmdb_id': str})
    d['genres'] = d['genres'].apply(lambda x: x.split('|') if isinstance(x, str) else [])
    c = Counter(g for gs in d['genres'] for g in gs if g in SCARCITY_ORDER)
    return c, len(d)

before, n_before = genre_counts(DATA_ROOT / 'labels.csv')     # v1 (dengesiz)
after,  n_after  = genre_counts(DATA_ROOT / 'labels_v2.csv')  # v2 (dengeli)

order = sorted(SCARCITY_ORDER, key=lambda g: after.get(g, 0))
y = np.arange(len(order)); h = 0.4
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(y + h/2, [before.get(g, 0) for g in order], height=h, label='v1 ham (n=%d)' % n_before)
ax.barh(y - h/2, [after.get(g, 0)  for g in order], height=h, label='v2 dengeli (n=%d)' % n_after)
ax.set_yticks(y); ax.set_yticklabels(order)
ax.set_xlabel('film sayisi'); ax.legend()
ax.set_title('Tur dagilimi: cekim oncesi (v1) vs sonrasi (v2)')
plt.tight_layout()
fig.savefig(DATA_ROOT / 'dist_before_after.png', dpi=120)
plt.show()
print('v2 max/min orani: %.2fx' % (max(after.values()) / min(after.values())))